# Lab 01 — Exploring LLM behaviour with OpenAI and Claude (solution)

> Reference solution: every TODO is filled in and every check passes.

**Level:** Beginner · **Duration:** ~60 minutes · **Covers:** environment check (15 min, setup) · H2 LLM behaviour exploration (45 min)

Lab guide: [`../lab-01-exploring-llm-behaviour.md`](../lab-01-exploring-llm-behaviour.md) — read each part there first; this notebook holds the code.

This lab is guided: every cell is given. You run it, observe, and record what you see in the observation tables. Model output varies from run to run, so record **your** results, not the guide's.

In [ ]:
# (notebook) Setup: run this cell first. It loads the repo-root .env and creates both SDK clients.
import json
import os
import sys
import time
from pathlib import Path

import anthropic
import openai
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env

# labs/module-01 holds data/ and templates/; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-01" and (p / "data").is_dir())
DATA_DIR = LAB_DIR / "data"
SETUP_DIR = LAB_DIR.parent / "setup"

# Model IDs come from .env, never from code (labs/setup/.env.example has the defaults).
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()
OPENAI_SMALL_MODEL = os.getenv("OPENAI_SMALL_MODEL", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "").strip()
ANTHROPIC_SMALL_MODEL = os.getenv("ANTHROPIC_SMALL_MODEL", "").strip()

# Each SDK reads its key from the environment; a client exists only when key and model are set.
openai_client = openai.OpenAI() if os.getenv("OPENAI_API_KEY") and OPENAI_MODEL else None
claude_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") and ANTHROPIC_MODEL else None


def available(provider: str) -> bool:
    """True if `provider` is configured; otherwise print why the cell skips it."""
    client = {"openai": openai_client, "anthropic": claude_client}[provider]
    if client is None:
        print(f"⚠️  {provider}: not configured, skipping. Add its key and model to .env, then restart the kernel.")
    return client is not None


def active_providers() -> list[str]:
    """The configured providers, with a printed note for each one that is skipped."""
    return [p for p in ("openai", "anthropic") if available(p)]


def claude_text(message) -> str:
    """Join the text blocks of a Claude response; thinking blocks are skipped."""
    return "".join(block.text for block in message.content if block.type == "text")


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(f"Lab folder : {LAB_DIR}")
print(f"OpenAI     : {'✅ ' + OPENAI_MODEL if openai_client else '❌ OPENAI_API_KEY or OPENAI_MODEL missing'}")
print(f"Claude     : {'✅ ' + ANTHROPIC_MODEL if claude_client else '❌ ANTHROPIC_API_KEY or ANTHROPIC_MODEL missing'}")
if not (openai_client or claude_client):
    print(
        "\n❌ No provider is configured. Copy labs/setup/.env.example to the repo-root .env, add your keys,\n"
        "   restart the kernel and run this cell again. Until then, cells that call a model are skipped."
    )

## Part 1 — Check the environment and make your first calls (15 min)

First the check script, then one request to each provider with the same prompt. Look at the
**shape** of each response: where the text is, how the call says it finished, and what usage it reports.

In [ ]:
# (notebook) Run the course check script for Module 1 (the same as the terminal command in the guide).
import subprocess

result = subprocess.run(
    [sys.executable, str(SETUP_DIR / "check_env.py"), "--module", "1"],
    capture_output=True,
    text=True,
)
print(result.stdout[-5000:] or result.stderr[-2000:])

In [ ]:
# (notebook) OpenAI: the Responses API is the primary path (Chat Completions still exists for older code).
PROMPT = "In two sentences, what does an L1 service-desk analyst do?"

oa_first = None
if available("openai"):
    started = time.perf_counter()
    oa_first = openai_client.responses.create(
        model=OPENAI_MODEL,
        input=PROMPT,
        reasoning={"effort": "low"},  # GPT-6 models reason before answering; low effort keeps this quick
        max_output_tokens=2_000,  # the cap covers reasoning tokens AND visible text
    )
    elapsed = time.perf_counter() - started
    print(oa_first.output_text, "\n")
    usage = oa_first.usage
    print(
        f"status={oa_first.status} · input={usage.input_tokens} · output={usage.output_tokens} "
        f"(reasoning {usage.output_tokens_details.reasoning_tokens}) · {elapsed:.1f}s · id={oa_first.id}"
    )

In [ ]:
# (notebook) Claude: the Messages API. max_tokens is required; effort lives in output_config.
cl_first = None
if available("anthropic"):
    started = time.perf_counter()
    cl_first = claude_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=2_000,  # the cap covers thinking AND visible text
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": PROMPT}],
    )
    elapsed = time.perf_counter() - started
    print(claude_text(cl_first), "\n")
    usage = cl_first.usage
    print(
        f"stop_reason={cl_first.stop_reason} · input={usage.input_tokens} · output={usage.output_tokens} "
        f"· {elapsed:.1f}s · id={cl_first.id}"
    )

In [ ]:
# (notebook) Compare the two response shapes side by side.
shapes = []
if oa_first is not None:
    shapes.append(
        {
            "provider": "OpenAI Responses",
            "output items": [item.type for item in oa_first.output],
            "text": "response.output_text",
            "finished": f"status={oa_first.status}",
            "usage": "input_tokens · output_tokens · output_tokens_details.reasoning_tokens",
        }
    )
if cl_first is not None:
    shapes.append(
        {
            "provider": "Claude Messages",
            "output items": [block.type for block in cl_first.content],
            "text": "text blocks in message.content",
            "finished": f"stop_reason={cl_first.stop_reason}",
            "usage": "input_tokens · output_tokens (thinking counts as output)",
        }
    )
pd.DataFrame(shapes)

In [ ]:
# (notebook) Two helpers wrap the calls you just made, so the rest of the lab stays short.
def ask_openai(prompt: str, *, effort: str = "low", max_output_tokens: int = 2_000) -> str:
    """Send one prompt to OPENAI_MODEL and return the visible text."""
    response = openai_client.responses.create(
        model=OPENAI_MODEL, input=prompt, reasoning={"effort": effort}, max_output_tokens=max_output_tokens
    )
    return response.output_text.strip()


def ask_claude(prompt: str, *, effort: str = "low", max_tokens: int = 2_000) -> str:
    """Send one prompt to ANTHROPIC_MODEL and return the visible text."""
    message = claude_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=max_tokens,
        output_config={"effort": effort},
        messages=[{"role": "user", "content": prompt}],
    )
    return claude_text(message).strip()


ASK = {"openai": ask_openai, "anthropic": ask_claude}

In [ ]:
# (notebook) Check: both first calls finished normally and returned text.
def check_part1():
    if oa_first is None and cl_first is None:
        return "no provider configured"
    if oa_first is not None:
        assert oa_first.status == "completed", f"OpenAI status is {oa_first.status}"
        assert oa_first.output_text.strip(), "OpenAI returned no visible text"
    if cl_first is not None:
        assert cl_first.stop_reason == "end_turn", f"Claude stop_reason is {cl_first.stop_reason}"
        assert claude_text(cl_first).strip(), "Claude returned no text block"


check("Part 1 — first calls", check_part1)

## Part 2 — Temperature and sampling (15 min)

A language model scores every token in its vocabulary (the *logits*), turns the scores into
probabilities with a softmax, and picks one. Temperature *T* divides the logits before the softmax:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

*T* < 1 sharpens the distribution, *T* > 1 flattens it, and greedy decoding always takes the top token.
You see it first in a 135M-parameter open model you can inspect completely, then in the hosted APIs.

In [ ]:
# (notebook) Load a small open model locally (cached by labs/setup/prefetch_models.py).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LOCAL_LM = "HuggingFaceTB/SmolLM2-135M"
NEXT_TOKEN_PROMPT = "If you forget your password, click the link to"
TEMPERATURES = (0.2, 0.7, 1.0, 1.5)

try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_LM)
    local_lm = AutoModelForCausalLM.from_pretrained(LOCAL_LM).eval()
except OSError as exc:
    tokenizer = local_lm = None
    print(f"⚠️  Could not load {LOCAL_LM} ({exc.__class__.__name__}). Run labs/setup/prefetch_models.py.")
    print("   Until then the cells below use an ILLUSTRATIVE logits vector, not a real model.")
else:
    print(f"Loaded {LOCAL_LM}: {sum(p.numel() for p in local_lm.parameters()) / 1e6:.0f}M parameters")

if local_lm is not None:
    with torch.no_grad():
        inputs = tokenizer(NEXT_TOKEN_PROMPT, return_tensors="pt")
        logits = local_lm(**inputs).logits[0, -1]  # one score per vocabulary entry
else:
    FALLBACK_TOKENS = [" reset", " the", " log", " change", " get", " your"]
    logits = torch.tensor([4.1, 3.6, 1.6, 0.0, 0.0, -0.5])  # illustrative only
print(f"Prompt: {NEXT_TOKEN_PROMPT!r} → {logits.shape[0]:,} candidate next tokens")

In [ ]:
# (notebook) The same logits under four temperatures: probability of the six most likely next tokens.
def token_label(index: int) -> str:
    return repr(tokenizer.decode(int(index))) if tokenizer is not None else repr(FALLBACK_TOKENS[int(index)])


def next_token_table(logits: torch.Tensor, temperatures, k: int = 6) -> pd.DataFrame:
    top_ids = torch.topk(logits, k).indices
    columns = {f"T={t}": torch.softmax(logits / t, dim=-1)[top_ids].tolist() for t in temperatures}
    return pd.DataFrame(columns, index=[token_label(i) for i in top_ids]).round(3)


def entropy(probs: torch.Tensor) -> float:
    """Shannon entropy in nats: 0 means certain, higher means more spread out."""
    return float(-(probs * torch.log(probs + 1e-12)).sum())


for t in TEMPERATURES:
    probs = torch.softmax(logits / t, dim=-1)
    print(f"T={t}: entropy {entropy(probs):5.2f} nats · top-1 probability {float(probs.max()):.3f}")
next_token_table(logits, TEMPERATURES)

In [ ]:
# (notebook) Sample 20 short continuations at each temperature and count how many are distinct.
def sample_continuations(temperature: float | None, n: int = 20, seed: int = 7) -> list[str]:
    """Generate n continuations of NEXT_TOKEN_PROMPT; temperature=None means greedy decoding."""
    torch.manual_seed(seed)
    inputs = tokenizer(NEXT_TOKEN_PROMPT, return_tensors="pt")
    if temperature is None:
        sampling = {"do_sample": False}
    else:  # top_k=0 and top_p=1.0 switch off the other filters, so only temperature acts
        sampling = {"do_sample": True, "temperature": temperature, "top_k": 0, "top_p": 1.0}
    outputs = local_lm.generate(
        **inputs,
        max_new_tokens=8,
        num_return_sequences=1 if temperature is None else n,
        pad_token_id=tokenizer.eos_token_id,
        **sampling,
    )
    start = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[start:], skip_special_tokens=True) for o in outputs]


diversity = {}
if local_lm is not None:
    for t in (None, 0.3, 1.0, 1.5):
        name = "greedy" if t is None else f"T={t}"
        continuations = sample_continuations(t)
        diversity[name] = len(set(continuations))
        print(f"{name:>7}: {diversity[name]:>2} distinct of {len(continuations):>2} · e.g. {continuations[-1]!r}")

### What the hosted APIs let you control in 2026

| | OpenAI Responses API | Claude Messages API |
| --- | --- | --- |
| `temperature`, `top_p` | Accepted only when `reasoning.effort` is `"none"`, which only some models support (`gpt-6.1-terra`, `gpt-6-luna`). Any other effort → **400**. | The Python SDK (1.x) has no `temperature` argument. Current models (Claude 4.7 and later) accept only the default; API version `2026-10-21` removes the fields altogether. |
| Token probabilities | `top_logprobs` + `include=["message.output_text.logprobs"]`, again only with effort `"none"` | Not exposed |
| What to use instead | Constrained tasks, structured outputs, validation, evals | The same |

The next two cells send a temperature anyway: the **deliberate failure** of this lab.

In [ ]:
# (notebook) Deliberate failure 1: temperature on a reasoning model at its default effort.
if available("openai"):
    try:
        openai_client.responses.create(model=OPENAI_MODEL, input="Say hello.", temperature=0.2, max_output_tokens=200)
        print(f"{OPENAI_MODEL} accepted temperature at its default reasoning effort.")
    except openai.BadRequestError as exc:
        print(f"❌ {OPENAI_MODEL} → 400 BadRequestError: {exc.message}")

In [ ]:
# (notebook) Deliberate failure 2: temperature on Claude, first through the SDK, then forced onto the wire.
if available("anthropic"):
    hello = [{"role": "user", "content": "Say hello."}]
    try:
        claude_client.messages.create(model=ANTHROPIC_MODEL, max_tokens=200, messages=hello, temperature=0.2)
    except TypeError as exc:
        print(f"❌ The SDK refuses before sending anything: TypeError: {exc}")
    try:
        claude_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=200, messages=hello, extra_body={"temperature": 0.2}
        )
        print(f"{ANTHROPIC_MODEL} accepted temperature=0.2.")
    except anthropic.BadRequestError as exc:
        print(f"❌ {ANTHROPIC_MODEL} → 400 BadRequestError: {exc.message}")

In [ ]:
# (notebook) Where temperature still works: OPENAI_SMALL_MODEL with reasoning effort "none".
import math

SWEEP_PROMPT = "Write a one-line status-page headline for a 20-minute VPN outage that is now resolved."


def first_token_alternatives(response, k: int = 5) -> list[tuple[str, float]]:
    """The k most likely first tokens and their probabilities, from the response logprobs."""
    for item in response.output:
        if item.type == "message":
            for part in item.content:
                if part.type == "output_text" and part.logprobs:
                    return [(alt.token, round(math.exp(alt.logprob), 3)) for alt in part.logprobs[0].top_logprobs[:k]]
    return []


sweep = {}
if not OPENAI_SMALL_MODEL:
    print("⚠️  OPENAI_SMALL_MODEL is not set: skipping the temperature sweep.")
elif available("openai"):
    try:
        for t in (0.0, 1.0, 1.8):
            texts = []
            for _ in range(4):
                response = openai_client.responses.create(
                    model=OPENAI_SMALL_MODEL,
                    input=SWEEP_PROMPT,
                    reasoning={"effort": "none"},
                    temperature=t,
                    max_output_tokens=60,
                    include=["message.output_text.logprobs"],
                    top_logprobs=5,
                )
                texts.append(response.output_text.strip())
            sweep[t] = texts
            print(f"T={t}: {len(set(texts))} distinct of {len(texts)} · first-token alternatives {first_token_alternatives(response)}")
            for text in texts:
                print("     ", text)
    except openai.BadRequestError as exc:
        print(f"❌ {OPENAI_SMALL_MODEL} rejected the sweep: {exc.message}")
        print("   OPENAI_SMALL_MODEL must support reasoning effort 'none' (see labs/setup/.env.example).")

In [ ]:
# (notebook) Default sampling on both main models: a constrained task versus an open one, 3 runs each.
TASKS = {
    "constrained": (
        "Classify the priority of this ticket as exactly one of P1, P2, P3, P4. Reply with the label only.\n"
        "Ticket: The whole EU-West office has no network access and payroll cannot run."
    ),
    "open": SWEEP_PROMPT,
}
RUNS = 3

variability = []
for provider in active_providers():
    for task, prompt in TASKS.items():
        outputs = [ASK[provider](prompt) for _ in range(RUNS)]
        variability.append(
            {"provider": provider, "task": task, "distinct": len(set(outputs)), "runs": RUNS, "first output": outputs[0][:90]}
        )
pd.DataFrame(variability)

📝 **Record (Part 2).** Fill in from your own run.

| Observation | Local SmolLM2 | OpenAI | Claude |
| ----------- | ------------- | ------ | ------ |
| Top-1 probability at T=0.2 vs T=1.5 | | n/a | n/a |
| Distinct outputs: constrained task | n/a | | |
| Distinct outputs: open task | n/a | | |
| What happened when you sent `temperature` | n/a | | |

In [ ]:
# (notebook) Check: the temperature mechanics behave as the formula says.
def check_part2():
    probs = [torch.softmax(logits / t, dim=-1) for t in TEMPERATURES]
    assert all(abs(float(p.sum()) - 1.0) < 1e-4 for p in probs), "probabilities must sum to 1"
    top1 = [float(p.max()) for p in probs]
    assert top1 == sorted(top1, reverse=True), "a lower temperature must give the top token more probability"
    if diversity:
        assert diversity["greedy"] == 1, "greedy decoding must be deterministic"
        assert diversity["T=1.5"] >= diversity["T=0.3"], "a higher temperature should not reduce diversity"


check("Part 2 — temperature mechanics", check_part2)

## Part 3 — Hallucination triggers and grounding (15 min)

Five prompts that commonly trigger confident, wrong answers. Run them on both providers, judge each
answer against the ground truth, then see how grounding (context + permission to abstain + a verbatim
quote you can check in code) changes the picture.

In [ ]:
# (notebook) The probe set.
probes = json.loads((DATA_DIR / "hallucination_probes.json").read_text(encoding="utf-8"))
pd.DataFrame(probes)[["id", "trigger", "prompt"]]

In [ ]:
# (notebook) Run every probe on every configured provider and read the answers in full.
import textwrap

probe_answers = []
providers = active_providers()
for probe in probes:
    row = {"id": probe["id"], "trigger": probe["trigger"]}
    for provider in providers:
        row[provider] = ASK[provider](probe["prompt"])
    probe_answers.append(row)

for probe, row in zip(probes, probe_answers):
    print(f"\n=== {probe['id']} · {probe['trigger']}\nPROMPT  {probe['prompt']}\nTRUTH   {probe['ground_truth']}")
    for provider in providers:
        print(f"--- {provider}\n" + textwrap.fill(row[provider], width=110, max_lines=8, placeholder=" […]"))

In [ ]:
# (notebook) Verify what code can verify: do the cited DOIs exist, and is the version current?
import importlib.metadata
import re

import httpx

DOI_PATTERN = re.compile(r"10\.\d{4,9}/[^\s\"'<>\])]+")


def doi_exists(doi: str) -> bool | None:
    """True or False from the doi.org handle API; None if doi.org cannot be reached."""
    try:
        reply = httpx.get(f"https://doi.org/api/handles/{doi.rstrip('.,;')}", timeout=5)
    except httpx.HTTPError:
        return None
    return reply.status_code == 200


def latest_pypi_version(package: str) -> str | None:
    try:
        return httpx.get(f"https://pypi.org/pypi/{package}/json", timeout=5).json()["info"]["version"]
    except (httpx.HTTPError, KeyError, ValueError):
        return None


citations = next((r for r in probe_answers if r["id"] == "H3"), {})
for provider in providers:
    dois = DOI_PATTERN.findall(citations.get(provider, ""))
    verdicts = {doi: doi_exists(doi) for doi in dois}
    print(f"{provider}: {len(dois)} DOI(s) → {verdicts or 'none given'}")

live = latest_pypi_version("anthropic")
installed = importlib.metadata.version("anthropic")
print(f"\nanthropic on PyPI now: {live or 'unreachable'} · installed in the course venv: {installed} (pinned on purpose)")
version_answers = next((r for r in probe_answers if r["id"] == "H5"), {})
for provider in providers:
    print(f"{provider} says: {version_answers.get(provider, '')[:40]!r}")

📝 **Record (Part 3).** For each probe: ✅ grounded behaviour, ⚠️ hedged but partly invented, ❌ confident and wrong.

| Probe | OpenAI | Claude | What gave it away? |
| ----- | ------ | ------ | ------------------ |
| H1 false premise | | | |
| H2 unseen internal document | | | |
| H3 citation request | | | |
| H4 unknowable private fact | | | |
| H5 knowledge cutoff | | | |

In [ ]:
# (notebook) Grounding: give the article, allow abstention, and demand a verbatim quote you can check.
from pydantic import BaseModel, Field

ARTICLE = (DATA_DIR / "kb" / "KB-0042.md").read_text(encoding="utf-8")


class GroundedAnswer(BaseModel):
    supported: bool = Field(description="True only if the article states the answer")
    answer: str | None = Field(description="The answer, or null when the article does not contain it")
    quote: str | None = Field(description="One sentence copied verbatim from the article that supports the answer, or null")


GROUNDING_INSTRUCTIONS = (
    "Answer the question using ONLY the knowledge-base article below. If the article does not contain "
    "the answer, set supported to false and answer and quote to null. Never use outside knowledge.\n\n"
    f"<article>\n{ARTICLE}\n</article>"
)
QUESTIONS = {
    "answerable": "What should I do first if the VPN client rejects my new password?",
    "unanswerable": "What is the Service Desk's out-of-hours phone number?",
}


def grounded_openai(question: str) -> GroundedAnswer:
    response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions=GROUNDING_INSTRUCTIONS,
        input=question,
        text_format=GroundedAnswer,  # strict JSON schema generated from the Pydantic model
        reasoning={"effort": "low"},
        max_output_tokens=2_000,
    )
    return response.output_parsed


def grounded_claude(question: str) -> GroundedAnswer:
    message = claude_client.messages.parse(
        model=ANTHROPIC_MODEL,
        max_tokens=2_000,
        system=GROUNDING_INSTRUCTIONS,
        messages=[{"role": "user", "content": question}],
        output_format=GroundedAnswer,  # the SDK turns it into output_config.format
        output_config={"effort": "low"},
    )
    return message.parsed_output


def normalise(text: str) -> str:
    return " ".join(text.split()).lower()


def verify_grounding(result: GroundedAnswer, article: str) -> str:
    """Deterministic check: a supported answer must quote the article verbatim."""
    if not result.supported:
        return "abstained"
    if result.quote and normalise(result.quote) in normalise(article):
        return "grounded (quote found)"
    return "UNVERIFIED (quote not in article)"


GROUNDED = {"openai": grounded_openai, "anthropic": grounded_claude}
grounding = []
for label, question in QUESTIONS.items():
    for provider in active_providers():
        result = GROUNDED[provider](question)
        grounding.append(
            {
                "question": label,
                "provider": provider,
                "supported": result.supported,
                "verdict": verify_grounding(result, ARTICLE),
                "answer": (result.answer or "")[:100],
            }
        )
pd.DataFrame(grounding)

In [ ]:
# (notebook) Check: every grounded answer parsed into the schema and got a verdict.
VERDICTS = {"abstained", "grounded (quote found)", "UNVERIFIED (quote not in article)"}


def check_part3():
    assert len(probes) >= 5, "the probe set should have at least five probes"
    assert len(probe_answers) == len(probes)
    if not grounding:
        return "no provider configured"
    assert all(isinstance(row["supported"], bool) for row in grounding)
    assert {row["verdict"] for row in grounding} <= VERDICTS
    for row in grounding:
        if row["question"] == "unanswerable" and row["supported"]:
            print(f"   ⚠️  {row['provider']} claimed the article answers the unanswerable question: verdict {row['verdict']}")


check("Part 3 — probes and grounding", check_part3)

## Part 4 — Context-window edge cases (15 min)

Four edges: how big the window really is, how many tokens your text costs (it depends on the
language and the tokeniser), what happens when the **output** budget runs out, and what a long
prompt does to retrieval and counting. You finish with a pre-flight guard that refuses a request
before it is sent.

In [ ]:
# (notebook) The limits: Claude reports them through the Models API; OpenAI's come from its docs.
catalogue = {m["id"]: m for m in json.loads((DATA_DIR / "model_catalogue.json").read_text())["models"]}

limits = []
if available("anthropic"):
    info = claude_client.models.retrieve(ANTHROPIC_MODEL)
    limits.append(
        {"model": info.id, "source": "Models API", "max input tokens": info.max_input_tokens, "max output tokens": info.max_tokens}
    )
if OPENAI_MODEL:
    entry = catalogue.get(OPENAI_MODEL)
    limits.append(
        {
            "model": OPENAI_MODEL,
            "source": "docs, via data/model_catalogue.json" if entry else "unknown: add it to the catalogue",
            "max input tokens": entry["max_input_tokens"] if entry else None,
            "max output tokens": entry["max_output_tokens"] if entry else None,
        }
    )
pd.DataFrame(limits)

In [ ]:
# (notebook) The same ticket in four languages: characters versus tokens.
import tiktoken

tickets = pd.read_csv(DATA_DIR / "multilingual_tickets.csv")
same_issue = tickets[tickets.gold_kb == "KB-0042"].reset_index(drop=True)

try:
    o200k = tiktoken.get_encoding("o200k_base")  # an offline approximation, cached by prefetch_models.py
except Exception as exc:
    o200k = None
    print(f"⚠️  o200k_base unavailable ({exc.__class__.__name__}); the offline column is skipped.")

if OPENAI_MODEL:
    try:
        print("tiktoken maps", OPENAI_MODEL, "to", tiktoken.encoding_for_model(OPENAI_MODEL).name)
    except KeyError:
        print(f"tiktoken does not know {OPENAI_MODEL}'s tokeniser → use the provider's count endpoint.")


def count_openai(text: str) -> int:
    return openai_client.responses.input_tokens.count(model=OPENAI_MODEL, input=text).input_tokens


def count_claude(text: str) -> int:
    return claude_client.messages.count_tokens(
        model=ANTHROPIC_MODEL, messages=[{"role": "user", "content": text}]
    ).input_tokens


rows = []
for ticket in same_issue.itertuples():
    row = {"language": ticket.language, "characters": len(ticket.text)}
    if o200k is not None:
        row["o200k_base (offline)"] = len(o200k.encode(ticket.text))
    if openai_client:
        row["OpenAI count API"] = count_openai(ticket.text)
    if claude_client:
        row["Claude count API"] = count_claude(ticket.text)
    rows.append(row)
token_table = pd.DataFrame(rows).set_index("language")
token_table

In [ ]:
# (notebook) The output budget runs out: reasoning/thinking tokens share the same cap as visible text.
LONG_ASK = "Explain, in about 300 words, how a VPN gateway authenticates a remote user."
truncation = []
if available("openai"):
    try:
        r = openai_client.responses.create(model=OPENAI_MODEL, input=LONG_ASK, max_output_tokens=64)
        truncation.append(
            {
                "provider": "openai",
                "finished": f"status={r.status} · reason={r.incomplete_details.reason if r.incomplete_details else None}",
                "output tokens": r.usage.output_tokens,
                "of which reasoning": r.usage.output_tokens_details.reasoning_tokens,
                "visible characters": len(r.output_text),
            }
        )
    except openai.BadRequestError as exc:
        print("OpenAI rejected the request:", exc.message)
if available("anthropic"):
    try:
        m = claude_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=64, messages=[{"role": "user", "content": LONG_ASK}]
        )
        details = m.usage.output_tokens_details
        truncation.append(
            {
                "provider": "anthropic",
                "finished": f"stop_reason={m.stop_reason} · blocks={[b.type for b in m.content]}",
                "output tokens": m.usage.output_tokens,
                "of which reasoning": details.thinking_tokens if details else None,
                "visible characters": len(claude_text(m)),
            }
        )
    except anthropic.BadRequestError as exc:
        print("Claude rejected the request:", exc.message)
pd.DataFrame(truncation)

In [ ]:
# (notebook) A synthetic operations log with one "needle" line, placed at the start, middle or end.
import random

HAYSTACK_LINES = 600  # about 15K tokens; lower it if your budget is tight
NEEDLE = (
    "2026-09-14 02:00 CHG-20931 approved by the change advisory board: core-switch firmware upgrade, "
    "maintenance window 02:00-04:00 UTC Sunday."
)
TEAMS = ["Service Desk", "Network Ops", "Platform Engineering", "Finance", "Legal"]
EVENTS = [
    "password reset completed", "VPN tunnel flapped", "disk usage above 85%", "printer queue cleared",
    "certificate renewed", "backup job finished", "MFA push timed out", "patch window opened",
]


def build_ops_log(lines: int, needle: str, position: float, seed: int = 42) -> tuple[str, dict[str, int]]:
    """Return (log text, P1 count per team) with `needle` inserted at `position` (0.0-1.0)."""
    rng = random.Random(seed)
    rows, p1_by_team = [], {team: 0 for team in TEAMS}
    for i in range(lines):
        team = rng.choice(TEAMS)
        priority = rng.choices(["P1", "P2", "P3", "P4"], weights=[1, 3, 8, 8])[0]
        p1_by_team[team] += priority == "P1"
        day = 1 + i * 29 // lines
        stamp = f"2026-09-{day:02d} {rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}"
        rows.append(f"{stamp} INC-{105000 + i} {priority} [{team}] {rng.choice(EVENTS)}")
    rows.insert(int(position * lines), needle)
    return "\n".join(rows), p1_by_team


sample_log, p1_by_team = build_ops_log(HAYSTACK_LINES, NEEDLE, 0.5)
approx_tokens = len(o200k.encode(sample_log)) if o200k is not None else len(sample_log) // 4
print(sample_log.splitlines()[0], "\n…")
print(f"{HAYSTACK_LINES} lines ≈ {approx_tokens:,} tokens per prompt (o200k_base estimate)")
for model_id in (OPENAI_MODEL, ANTHROPIC_MODEL):
    if model_id in catalogue and catalogue[model_id]["input"]:
        cost = 4 * approx_tokens * catalogue[model_id]["input"] / 1_000_000
        print(f"  the next cell sends 4 such prompts to {model_id}: ≈ ${cost:.2f} of input")

In [ ]:
# (notebook) Needle retrieval at three positions, then a counting question across the whole log.
NEEDLE_QUESTION = "Which change request did the change advisory board approve, and what is its maintenance window? Answer in one line."
COUNT_QUESTION = "How many P1 incidents for [Network Ops] are in the log? Reply with the number only."

needle_results = []
providers = active_providers()
for position in (0.05, 0.5, 0.95):
    log, _ = build_ops_log(HAYSTACK_LINES, NEEDLE, position)
    prompt = f"<ops_log>\n{log}\n</ops_log>\n\n{NEEDLE_QUESTION}"
    for provider in providers:
        answer = ASK[provider](prompt)
        found = all(token in answer for token in ("CHG-20931", "02:00", "04:00"))
        needle_results.append({"position": position, "provider": provider, "found": found, "answer": answer[:90]})

truth = p1_by_team["Network Ops"]
counting = []
for provider in providers:
    answer = ASK[provider](f"<ops_log>\n{sample_log}\n</ops_log>\n\n{COUNT_QUESTION}")
    number = re.search(r"\d+", answer)
    counting.append({"provider": provider, "answer": answer[:30], "correct": bool(number) and int(number.group()) == truth})

counted_by_code = sum(" P1 [Network Ops] " in line for line in sample_log.splitlines())
print(f"Ground truth from code: {truth} P1 incidents for Network Ops (a one-line scan finds {counted_by_code})")
display(pd.DataFrame(needle_results))
pd.DataFrame(counting)

In [ ]:
# (notebook) A pre-flight guard: count first, refuse before sending if the request cannot fit.
class ContextBudgetError(RuntimeError):
    """The prompt plus the reserved output would exceed the context budget."""


def count_tokens(provider: str, text: str) -> int:
    """Count with the provider's own endpoint; fall back to o200k_base offline."""
    if provider == "openai" and openai_client:
        return count_openai(text)
    if provider == "anthropic" and claude_client:
        return count_claude(text)
    if o200k is not None:
        return len(o200k.encode(text))
    raise RuntimeError("no token counter available")


def preflight(provider: str, prompt: str, *, max_output_tokens: int, budget_tokens: int) -> int:
    """Return the tokens the request needs, or raise ContextBudgetError if it would not fit."""
    needed = count_tokens(provider, prompt) + max_output_tokens
    if needed > budget_tokens:
        raise ContextBudgetError(
            f"{provider}: needs {needed:,} tokens (prompt + {max_output_tokens:,} reserved for output) "
            f"but the budget is {budget_tokens:,}"
        )
    return needed


# An 8K budget stands in for a small model or a cost cap; the real windows are 1M tokens.
if o200k is None and not providers:
    print("⚠️  No token counter available (no provider, no cached o200k_base): skipping the guard demo.")
for provider in providers or (["offline"] if o200k is not None else []):
    try:
        needed = preflight(provider, sample_log, max_output_tokens=2_000, budget_tokens=8_000)
        print(f"{provider}: fits ({needed:,} tokens)")
    except ContextBudgetError as exc:
        print("🛑", exc)

📝 **Record (Part 4).**

| Observation | OpenAI | Claude |
| ----------- | ------ | ------ |
| Tokens for the Hindi ticket ÷ tokens for the English one | | |
| How the truncated call reported it, and how much visible text came back | | |
| Needle found at 5% / 50% / 95% | | |
| P1 count correct? | | |

When a real request is too big, you **trim** (drop old turns), **summarise**, or **retrieve** only the
relevant part (Module 3). Claude reports `stop_reason="model_context_window_exceeded"` when generation
hits the window; the Responses API has `truncation="auto"` to drop the oldest conversation items instead of failing.

In [ ]:
# (notebook) Check: the deterministic parts of Part 4.
def check_part4():
    log, counts = build_ops_log(100, NEEDLE, 0.5)
    assert log.splitlines()[50] == NEEDLE, "the needle must sit at the requested position"
    assert counts["Network Ops"] == sum(" P1 [Network Ops] " in line for line in log.splitlines())
    if o200k is None and not providers:
        return "no token counter available"
    try:
        preflight((providers or ["offline"])[0], sample_log, max_output_tokens=100, budget_tokens=10)
    except ContextBudgetError:
        pass
    else:
        raise AssertionError("preflight must refuse a request larger than the budget")
    assert (token_table.select_dtypes("number") > 0).all().all(), "every token count must be positive"


check("Part 4 — context edges", check_part4)

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 02 — Document intelligence: OCR and multilingual embeddings](../lab-02-ocr-multilingual-embeddings.md).